In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.transforms.v2.functional as TF
import matplotlib.pyplot as plt

from datasets import load_dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "xpu" if torch.xpu.is_available() else "cpu")

c:\Users\vivaa\workingfolder\college\sem 3\dbms\question_creation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dataset = load_dataset("ylecun/mnist")

# Resize images to 15x15 and convert to tensor
def transform(example):
    example["image"] = [k.resize((15, 15)) for k in example["image"]]
    example["image"] = [TF.to_tensor(k) for k in example["image"]]
    return example
dataset = dataset.with_transform(transform)

train_dataloader = DataLoader(dataset["train"], batch_size=64, shuffle=True)
test_dataloader = DataLoader(dataset["test"], batch_size=64, shuffle=False)

In [4]:
dataset["train"][0]["image"]

c:\Users\vivaa\workingfolder\college\sem 3\dbms\question_creation\.venv\Lib\site-packages\torchvision\transforms\v2\functional\_deprecated.py:12: UserWarning: The function `to_tensor(...)` is deprecated and will be removed in a future release. Instead, please use `to_image(...)` followed by `to_dtype(..., dtype=torch.float32, scale=True)`.
  warnings.warn(


tensor([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0549, 0.1804, 0.1176, 0.3176, 0.1961, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0039, 0.2392, 0.4275, 0.6078, 0.8000,
          0.8706, 0.8627, 0.6118, 0.8235, 0.3569, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0039, 0.6549, 0.9765, 1.0000, 0.8549,
          0.7725, 0.6353, 0.0824, 0.0431, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0863, 0.2549, 0.8824, 0.2902,
          0.0000, 0.1098, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0039, 0.6039, 0.5686,
          0.0275, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0

In [5]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(15 * 15, 10)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(10, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

In [6]:
def train(model, train_dataloader, device, epochs=5):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    loss_history = []

    model.train()
    for epoch in range(epochs):
        epoch_loss_h = []
        print(f"Epoch: {epoch + 1}/{epochs}")
        for batch in train_dataloader:
            inputs = batch['image'].to(device)
            labels = batch['label'].to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss_h.append(loss.item())
        print(f"Loss: {sum(epoch_loss_h)/len(epoch_loss_h):.4f}")
        loss_history.extend(epoch_loss_h)

    return loss_history

In [7]:
model = MLP()
model.load_state_dict(torch.load("models/mlp_model_15.pth", map_location=DEVICE))
model.to(DEVICE)

# loss_history = train(model, train_dataloader, DEVICE, epochs=10)

# plt.plot(loss_history)
# plt.title("Training Loss")
# plt.xlabel("Batch")
# plt.ylabel("Loss")

# plt.show()

MLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=225, out_features=10, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=10, out_features=10, bias=True)
)

In [8]:
# Test the model
model.eval()
for batch in test_dataloader:
    inputs = batch['image'].to(DEVICE)
    labels = batch['label'].to(DEVICE)

    outputs = model(inputs)
    _, predicted = torch.max(outputs, 1)

    print(f"Predicted: {predicted.cpu().numpy()}")
    print(f"Actual: {labels.cpu().numpy()}")
    break  # Just show the first batch

# Calculate average accuracy on the test set
correct = 0
total = 0
model.eval()
for batch in test_dataloader:
    inputs = batch['image'].to(DEVICE)
    labels = batch['label'].to(DEVICE)

    outputs = model(inputs)
    _, predicted = torch.max(outputs, 1)

    total += labels.size(0)
    correct += (predicted == labels).sum().item()

print(f"Accuracy on the test set: {100 * correct / total:.2f}%")

Predicted: [7 2 1 0 4 1 4 9 6 9 0 6 9 0 1 5 9 7 3 4 9 6 6 5 4 0 7 4 0 1 3 1 3 6 7 2 7
 1 2 1 1 7 4 2 3 5 1 2 4 4 6 3 5 5 6 0 4 1 9 5 7 8 9 3]
Actual: [7 2 1 0 4 1 4 9 5 9 0 6 9 0 1 5 9 7 3 4 9 6 6 5 4 0 7 4 0 1 3 1 3 4 7 2 7
 1 2 1 1 7 4 2 3 5 1 2 4 4 6 3 5 5 6 0 4 1 9 5 7 8 9 3]
Accuracy on the test set: 93.24%


In [9]:
# Save the model
torch.save(model.state_dict(), "models/mlp_model_15.pth")

In [10]:
import torchinfo
torchinfo.summary(model, input_size=(1, 1, 15, 15), device=DEVICE)

Layer (type:depth-idx)                   Output Shape              Param #
MLP                                      [1, 10]                   --
├─Flatten: 1-1                           [1, 225]                  --
├─Linear: 1-2                            [1, 10]                   2,260
├─ReLU: 1-3                              [1, 10]                   --
├─Linear: 1-4                            [1, 10]                   110
Total params: 2,370
Trainable params: 2,370
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.01
Estimated Total Size (MB): 0.01

In [ ]:
import pandas as pd

pd.DataFrame(model.fc1.weight.data.T.detach().cpu().numpy()).to_csv('layers_mlp/w1.csv')
pd.DataFrame(model.fc1.bias.data.T.detach().cpu().numpy()).to_csv('layers_mlp/b1.csv')
pd.DataFrame(model.fc2.weight.data.T.detach().cpu().numpy()).to_csv('layers_mlp/w2.csv')
pd.DataFrame(model.fc2.bias.data.T.detach().cpu().numpy()).to_csv('layers_mlp/b2.csv')

C:\Users\vivaa\AppData\Local\Temp\ipykernel_34400\1266491702.py:4: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4323.)
  pd.DataFrame(model.fc1.bias.data.T.detach().cpu().numpy()).to_csv('layers/b1.csv')


In [14]:
model.fc2.bias.data.shape

torch.Size([10])